# MLX portable tracking replay

Explore `replay.json` without the detector checkpoint or source video. The notebook plots trajectories and builds a 2D bounding-box animation from the provider-neutral export.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib import animation, patches
import pandas as pd
from IPython.display import HTML, display

RESULT_DIR = Path("../results/libreyolo-tracking-final").resolve()
REPLAY_PATH = RESULT_DIR / "replay.json"
payload = json.loads(REPLAY_PATH.read_text(encoding="utf-8"))
print(f"Loaded {REPLAY_PATH}")

In [ ]:
predictions = pd.DataFrame(payload["predictions"]["records"])
ground_truth = pd.DataFrame(
    payload["ground_truth"]["records"] if payload["ground_truth"] else []
)
metrics = pd.Series(payload["metrics"] or {}, name="value").to_frame()
display(metrics)
display(predictions.head())

## Trajectory projection

Each line follows the center of one predicted bounding box. Image coordinates use a top-left origin, so the y-axis is inverted.

In [ ]:
trajectory = predictions.assign(
    center_x=predictions.left + predictions.width / 2,
    center_y=predictions.top + predictions.height / 2,
)
fig, ax = plt.subplots(figsize=(12, 8))
for track_id, rows in trajectory.groupby("track_id"):
    ax.plot(rows.center_x, rows.center_y, linewidth=1.8, alpha=0.8, label=f"ID {track_id}")
ax.set(
    xlim=(0, payload["canvas"]["width"]),
    ylim=(payload["canvas"]["height"], 0),
    xlabel="x (pixels)", ylabel="y (pixels)", title="Predicted track trajectories",
)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(ncols=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.08))
plt.show()

## Video-free 2D replay

Solid colored boxes are predictions; dashed green boxes are ground truth. Track colors are stable within this notebook session.

In [ ]:
width, height = payload["canvas"]["width"], payload["canvas"]["height"]
frame_count, fps = payload["frame_count"], payload["fps"]
fig, ax = plt.subplots(figsize=(12, 8))

def draw_frame(frame_id):
    ax.clear()
    ax.set(xlim=(0, width), ylim=(height, 0), title=f"Frame {frame_id} / {frame_count}")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15)
    if not ground_truth.empty:
        for row in ground_truth[ground_truth.frame_id == frame_id].itertuples():
            ax.add_patch(patches.Rectangle(
                (row.left, row.top), row.width, row.height, fill=False,
                edgecolor="#22c55e", linewidth=1.8, linestyle="--",
            ))
            ax.text(row.left, row.top, f"GT {row.track_id}", color="#15803d", fontsize=8)
    for row in predictions[predictions.frame_id == frame_id].itertuples():
        color = plt.cm.hsv((row.track_id * 0.61803398875) % 1.0)
        ax.add_patch(patches.Rectangle(
            (row.left, row.top), row.width, row.height, fill=False,
            edgecolor=color, linewidth=2.2,
        ))
        ax.text(row.left, row.top, f"ID {row.track_id} · {row.confidence:.2f}", color=color, fontsize=8)

replay = animation.FuncAnimation(
    fig, draw_frame, frames=range(1, frame_count + 1),
    interval=1000 / fps, repeat=False,
)
plt.close(fig)
HTML(replay.to_jshtml(fps=fps))

The output directory also contains `replay.html`, which offers faster interactive playback and can be opened directly in a browser without Jupyter.